# Reproducing GLOT Paper Results with Original Code

This notebook runs the **original GLOT implementation** from [ipsitmantri/GLOT](https://github.com/ipsitmantri/GLOT) to reproduce the paper's reported results.

**Experiments:**
1. GLUE benchmark (9 tasks × 5 poolers × 2 encoder backbones)
2. Diagnostic stress test (4 distractor ratios × 5 poolers × 2 backbones)

**Hardware:** Requires GPU. Go to `Runtime → Change runtime type → T4 GPU` (or A100 for decoder models).

**Time estimate:** ~2-4 hours for all encoder experiments on T4.

In [ ]:
import torch, os, json, subprocess, time
from google.colab import drive, userdata

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Go to Runtime → Change runtime type → T4 GPU"
    )
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

# Mount Google Drive for persistent results
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/glot_reproduction"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

In [ ]:
import sys

ORG_REPO = "https://github.com/ipsitmantri/GLOT.git"
ORG_DIR = "/content/org_glot"

if not os.path.exists(ORG_DIR):
    !git clone {ORG_REPO} {ORG_DIR}
else:
    !cd {ORG_DIR} && git pull

# Show repo contents
!ls -la {ORG_DIR}

In [ ]:
# Detect Colab CUDA version and install compatible PyG
import subprocess
cuda_version = torch.version.cuda  # e.g. "12.4"
print(f"PyTorch CUDA: {cuda_version}")

# Install their requirements, but handle PyG wheels for Colab's CUDA
# Their requirements.txt pins torch-2.8.0+cu129 which may not match Colab
# We install PyG components compatible with the existing torch/CUDA

!pip install -q torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv \
    transformers datasets sentence-transformers mteb peft wandb accelerate \
    scikit-learn numpy pandas tqdm matplotlib seaborn polars

print("Dependencies installed")

In [ ]:
# HuggingFace token for gated models
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    HF_TOKEN = ""
    print("WARNING: No HF_TOKEN found. Gated models (Llama, Mistral) will fail.")
    print("Add your token in Colab: Settings (gear icon) → Secrets → HF_TOKEN")

# Patch the HF_TOKEN in main.py so it doesn't use the placeholder
main_py = os.path.join(ORG_DIR, "main.py")
with open(main_py, "r") as f:
    content = f.read()
content = content.replace('HF_TOKEN = "<>"', f'HF_TOKEN = os.environ.get("HF_TOKEN", "")')
with open(main_py, "w") as f:
    f.write(content)

# Set wandb to offline mode (no account needed)
os.environ["WANDB_MODE"] = "offline"
print("W&B set to offline mode (results captured from stdout)")

In [ ]:
# ============================================================
# Experiment Configuration
# ============================================================

# Encoder backbones (feasible on T4)
ENCODER_BACKBONES = [
    "bert-base-uncased",
    "roberta-base",
]

# Pooling methods to test
POOLERS = ["mean", "max", "cls", "adapool", "glot"]

# GLUE tasks — grouped by type
SINGLE_TASKS = ["sst2", "cola"]  # Single sentence classification
PAIR_TASKS = ["mrpc", "qqp", "stsb", "mnli", "qnli", "rte", "wnli"]  # Pair tasks
ALL_GLUE_TASKS = SINGLE_TASKS + PAIR_TASKS

# Diagnostic stress test ratios
DISTRACTOR_RATIOS = [0.2, 0.5, 0.8, 0.9]

# Hyperparameters (matching paper defaults)
HPARAMS = {
    "epochs": 3,
    "batch_size": 32,
    "eval_batch_size": 64,
    "lr": 2e-4,
    "max_length": 128,
    "seed": 0,
    "gat_hidden_dim": 256,
    "num_layers": 4,
    "jk_mode": "cat",
    "tau": 0.3,
    "scorer_hidden": 128,
    "precompute_hidden_states": 1,
    "verbose": 1,
}

# Results file path (persistent on Google Drive)
RESULTS_FILE = os.path.join(RESULTS_DIR, "glue_results.json")
DIAG_RESULTS_FILE = os.path.join(RESULTS_DIR, "diagnostic_results.json")

# Load existing results if resuming
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        glue_results = json.load(f)
    print(f"Loaded {len(glue_results)} existing GLUE results")
else:
    glue_results = {}

if os.path.exists(DIAG_RESULTS_FILE):
    with open(DIAG_RESULTS_FILE, "r") as f:
        diag_results = json.load(f)
    print(f"Loaded {len(diag_results)} existing diagnostic results")
else:
    diag_results = {}

print(f"Experiment matrix: {len(ENCODER_BACKBONES)} backbones × {len(POOLERS)} poolers × {len(ALL_GLUE_TASKS)} tasks = {len(ENCODER_BACKBONES) * len(POOLERS) * len(ALL_GLUE_TASKS)} GLUE experiments")
print(f"Diagnostic matrix: {len(ENCODER_BACKBONES)} backbones × {len(POOLERS)} poolers × {len(DISTRACTOR_RATIOS)} ratios = {len(ENCODER_BACKBONES) * len(POOLERS) * len(DISTRACTOR_RATIOS)} diagnostic experiments")

In [ ]:
import re

def run_glue_experiment(backbone, pooler, task, hparams, org_dir=ORG_DIR):
    """Run a single GLUE experiment using the original main.py and parse metrics from stdout."""
    key = f"{backbone}|{pooler}|{task}"

    # Skip if already done
    if key in glue_results:
        print(f"  SKIP (cached): {key} -> {glue_results[key]}")
        return glue_results[key]

    cmd = [
        "python", os.path.join(org_dir, "main.py"),
        "--model_name_or_path", backbone,
        "--task", task,
        "--pooling_method", pooler,
        "--epochs", str(hparams["epochs"]),
        "--batch_size", str(hparams["batch_size"]),
        "--eval_batch_size", str(hparams["eval_batch_size"]),
        "--lr", str(hparams["lr"]),
        "--max_length", str(hparams["max_length"]),
        "--seed", str(hparams["seed"]),
        "--gat_hidden_dim", str(hparams["gat_hidden_dim"]),
        "--num_layers", str(hparams["num_layers"]),
        "--jk_mode", hparams["jk_mode"],
        "--tau", str(hparams["tau"]),
        "--scorer_hidden", str(hparams["scorer_hidden"]),
        "--precompute_hidden_states", str(hparams["precompute_hidden_states"]),
        "--verbose", str(hparams["verbose"]),
        "--graph_adj", "threshold",
        "--weight_decay", "1e-5",
    ]

    print(f"  Running: {backbone} | {pooler} | {task} ...", end=" ", flush=True)
    start = time.time()

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=1800,  # 30 min timeout
            cwd=org_dir
        )
        elapsed = time.time() - start
        output = result.stdout + result.stderr

        # Parse metrics from verbose output
        # Format: [pooler] epoch N loss X.XXXX acc Y.YYYY mcc Z.ZZZZ
        # or: [pooler] epoch N MSE X.XXXX Spearman Y.YYYY Pearson Z.ZZZZ
        metrics = {}

        # Find all epoch lines and take the best
        acc_matches = re.findall(r"acc\s+([\d.]+)", output)
        mcc_matches = re.findall(r"mcc\s+([\d.]+)", output)
        f1_matches = re.findall(r"f1\s+([\d.]+)", output)
        spearman_matches = re.findall(r"Spearman\s+([\d.]+)", output)
        pearson_matches = re.findall(r"Pearson\s+([\d.]+)", output)

        if acc_matches:
            metrics["acc"] = max(float(x) for x in acc_matches)
        if mcc_matches:
            metrics["mcc"] = max(float(x) for x in mcc_matches)
        if f1_matches:
            metrics["f1"] = max(float(x) for x in f1_matches)
        if spearman_matches:
            metrics["spearman"] = max(float(x) for x in spearman_matches)
        if pearson_matches:
            metrics["pearson"] = max(float(x) for x in pearson_matches)

        metrics["elapsed_sec"] = round(elapsed, 1)
        metrics["returncode"] = result.returncode

        if result.returncode != 0:
            metrics["error"] = output[-500:]  # Last 500 chars of output
            print(f"FAILED ({elapsed:.0f}s)")
        else:
            print(f"done ({elapsed:.0f}s) -> {metrics}")

    except subprocess.TimeoutExpired:
        metrics = {"error": "TIMEOUT (30 min)", "elapsed_sec": 1800}
        print("TIMEOUT")
    except Exception as e:
        metrics = {"error": str(e)}
        print(f"ERROR: {e}")

    # Save result
    glue_results[key] = metrics
    with open(RESULTS_FILE, "w") as f:
        json.dump(glue_results, f, indent=2)

    return metrics

print("Experiment runner ready")

## GLUE Benchmark Experiments

Running all GLUE tasks with encoder backbones. Each experiment:
1. Precomputes hidden states from the frozen backbone
2. Trains the pooler head for the configured number of epochs
3. Evaluates on the validation set

Results are saved to Google Drive after each experiment for crash resilience.

In [ ]:
# ============================================================
# Run GLUE Experiments
# ============================================================

total = len(ENCODER_BACKBONES) * len(POOLERS) * len(ALL_GLUE_TASKS)
done = 0

for backbone in ENCODER_BACKBONES:
    print(f"\n{'='*60}")
    print(f"BACKBONE: {backbone}")
    print(f"{'='*60}")

    for task in ALL_GLUE_TASKS:
        print(f"\n--- Task: {task} ---")
        for pooler in POOLERS:
            done += 1
            print(f"[{done}/{total}]", end=" ")
            run_glue_experiment(backbone, pooler, task, HPARAMS)

print(f"\n\nAll GLUE experiments complete! Results saved to {RESULTS_FILE}")

In [ ]:
import pandas as pd

# Task -> primary metric mapping
TASK_METRICS = {
    "sst2": ("acc", "Acc"),
    "cola": ("mcc", "MCC"),
    "mrpc": ("f1", "F1"),
    "qqp": ("f1", "F1"),
    "stsb": ("spearman", "Spear."),
    "mnli": ("acc", "Acc"),
    "qnli": ("acc", "Acc"),
    "rte": ("acc", "Acc"),
    "wnli": ("acc", "Acc"),
}

for backbone in ENCODER_BACKBONES:
    print(f"\n{'='*80}")
    print(f"  {backbone}")
    print(f"{'='*80}")

    rows = []
    for pooler in POOLERS:
        row = {"Pooler": pooler.upper()}
        for task in ALL_GLUE_TASKS:
            key = f"{backbone}|{pooler}|{task}"
            metric_key, metric_label = TASK_METRICS[task]
            result = glue_results.get(key, {})

            if "error" in result:
                row[f"{task}\n({metric_label})"] = "ERR"
            elif metric_key in result:
                val = result[metric_key]
                # Convert to percentage if needed (acc/f1/mcc are in [0,1] from original code)
                if metric_key in ["acc", "f1", "mcc"]:
                    val *= 100
                row[f"{task}\n({metric_label})"] = f"{val:.1f}"
            else:
                row[f"{task}\n({metric_label})"] = "N/A"
        rows.append(row)

    df = pd.DataFrame(rows)
    df.set_index("Pooler", inplace=True)
    print(df.to_string())
    print()

## Diagnostic Stress Test

The "Relational Needle in a Haystack" experiment tests pooler robustness by burying signal phrases in random distractor words at varying ratios (20%, 50%, 80%, 90% noise).

In [ ]:
def run_diagnostic_experiment(backbone, pooler, ratio, org_dir=ORG_DIR):
    """Run a single diagnostic stress test experiment."""
    key = f"{backbone}|{pooler}|{ratio}"

    if key in diag_results:
        print(f"  SKIP (cached): {key} -> {diag_results[key]}")
        return diag_results[key]

    cmd = [
        "python", os.path.join(org_dir, "diagnostic_stress_test.py"),
        "--model_name_or_path", backbone,
        "--pooling_method", pooler,
        "--distractor_ratio", str(ratio),
        "--seed", str(HPARAMS["seed"]),
        "--gat_hidden_dim", str(HPARAMS["gat_hidden_dim"]),
        "--num_layers", str(HPARAMS["num_layers"]),
        "--jk_mode", HPARAMS["jk_mode"],
        "--tau", str(HPARAMS["tau"]),
        "--verbose", str(HPARAMS["verbose"]),
    ]

    print(f"  Running: {backbone} | {pooler} | ratio={ratio} ...", end=" ", flush=True)
    start = time.time()

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=600,  # 10 min timeout
            cwd=org_dir
        )
        elapsed = time.time() - start
        output = result.stdout + result.stderr

        metrics = {}
        acc_matches = re.findall(r"acc(?:uracy)?[\s:=]+([\d.]+)", output, re.IGNORECASE)
        if acc_matches:
            metrics["acc"] = max(float(x) for x in acc_matches)

        metrics["elapsed_sec"] = round(elapsed, 1)
        metrics["returncode"] = result.returncode

        if result.returncode != 0:
            metrics["error"] = output[-500:]
            print(f"FAILED ({elapsed:.0f}s)")
        else:
            print(f"done ({elapsed:.0f}s) -> {metrics}")

    except subprocess.TimeoutExpired:
        metrics = {"error": "TIMEOUT", "elapsed_sec": 600}
        print("TIMEOUT")
    except Exception as e:
        metrics = {"error": str(e)}
        print(f"ERROR: {e}")

    diag_results[key] = metrics
    with open(DIAG_RESULTS_FILE, "w") as f:
        json.dump(diag_results, f, indent=2)

    return metrics

print("Diagnostic runner ready")

In [ ]:
# ============================================================
# Run Diagnostic Stress Tests
# ============================================================

total = len(ENCODER_BACKBONES) * len(POOLERS) * len(DISTRACTOR_RATIOS)
done = 0

for backbone in ENCODER_BACKBONES:
    print(f"\n{'='*60}")
    print(f"BACKBONE: {backbone}")
    print(f"{'='*60}")

    for ratio in DISTRACTOR_RATIOS:
        print(f"\n--- Distractor Ratio: {ratio} ---")
        for pooler in POOLERS:
            done += 1
            print(f"[{done}/{total}]", end=" ")
            run_diagnostic_experiment(backbone, pooler, ratio)

print(f"\n\nAll diagnostic experiments complete! Results saved to {DIAG_RESULTS_FILE}")

In [ ]:
for backbone in ENCODER_BACKBONES:
    print(f"\n{'='*60}")
    print(f"  Diagnostic: {backbone}")
    print(f"{'='*60}")

    rows = []
    for pooler in POOLERS:
        row = {"Pooler": pooler.upper()}
        for ratio in DISTRACTOR_RATIOS:
            key = f"{backbone}|{pooler}|{ratio}"
            result = diag_results.get(key, {})
            if "error" in result:
                row[f"r={ratio}"] = "ERR"
            elif "acc" in result:
                row[f"r={ratio}"] = f"{result['acc'] * 100:.1f}"
            else:
                row[f"r={ratio}"] = "N/A"
        rows.append(row)

    df = pd.DataFrame(rows)
    df.set_index("Pooler", inplace=True)
    print(df.to_string())
    print()